In [ ]:
# 2025.11.30 LGBM HyperOpt & Train Result


In [1]:
import sys

project_root = 'c:/big20/git/big20-ML-project2-team3/CreditCardFraud'

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
import os

import time
import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt
import seaborn as sns


import warnings
warnings.filterwarnings('ignore')

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics         import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics         import roc_auc_score
from sklearn.metrics         import precision_recall_curve, classification_report
from sklearn.datasets         import make_classification

# Model import
from sklearn.tree           import DecisionTreeClassifier
from sklearn.ensemble       import RandomForestClassifier
from sklearn.ensemble       import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model   import LogisticRegression
from sklearn.linear_model   import LinearRegression
from sklearn.svm            import SVC
from sklearn.metrics        import classification_report
from xgboost                import XGBClassifier
from xgboost                import plot_importance
from lightgbm               import LGBMClassifier
from catboost               import CatBoostClassifier

# hyperopt 용
from hyperopt               import hp

# 사용자 Functions import
import HyperParams          as HP 
import utils.data_sampling  as ds 

import importlib

from utils import hyperopt_search
importlib.reload(hyperopt_search)


from utils import user_utils    as uu
from utils import preprocessing as pp
from utils import data_sampling as ds
from utils import model_utils   as mu
from utils import modeling      as mo

from utils.hyperopt_search import hyperopt_search, train_and_evaluate

In [3]:
# 결과받을 딕셔너리
results = {}
team_rs = 23 # 우리팀 random_state

In [4]:
#1. 데이터 로딩
raw_df = pp.ccf_load_data()

데이터 로드 성공: (284807, 31)


In [5]:
# 2. Time 컬럼 삭제 , 데이터,타겟 분리
X_features, y_target = pp.split_features_target(raw_df, cols= 'Time')

In [6]:
# 3. 이상치를 경계값으로 치환
cap_X_feature = pp.cap_outliers(X_features)

In [7]:
# 4.1 학습/테스트 데이터 분리
X_train, X_test, y_train, y_test = pp.data_split(cap_X_feature, y_target)

In [ ]:
# 5.1 학습/검증 데이터 분리
# X_tr, X_val, y_tr, y_val = pp.data_split(X_train, y_train, size=0.4)

In [8]:
# 4.2 Over Sampling 하는 경우
X_over, y_over = ds.oversampling_smote(X_train, y_train)


✅ SMOTE 오버샘플링 완료
   원본 샘플 수: 227845 (Class 0: 227451, Class 1: 394)
   샘플링 후: 454902 (Class 0: 227451, Class 1: 227451)


In [ ]:
# 5.2 Over Sampling한 경우 학습/검증 데이터 분리
# X_tr_over, X_val_over, y_tr_over, y_val_over = pp.data_split(X_over, y_over, size=0.4)

In [18]:
# 사용자 Functions import - 에러나서 다시 실행
import importlib
from utils import hyperopt_search
importlib.reload(hyperopt_search)

from utils.hyperopt_search import hyperopt_search, train_and_evaluate

In [ ]:
# 모델별 스페이스 생성 : LightGBM
lgbm_search_space = {
    # 학습률 (로그 스케일이 더 효과적)
    'learning_rate': hp.loguniform('learning_rate', np.log(0.01), np.log(0.3)),
    
    # 트리 구조
    'num_leaves': hp.quniform('num_leaves', 31, 255, 1),  # 2^n-1 권장
    'max_depth': hp.quniform('max_depth', 3, 12, 1),  # 범위 확장
    'n_estimators': hp.quniform('n_estimators', 100, 1000, 50),
    
    # Feature sampling
    'feature_fraction': hp.uniform('feature_fraction', 0.6, 1.0),  # 0.5는 너무 낮음
    'bagging_fraction': hp.uniform('bagging_fraction', 0.6, 1.0),
    'bagging_freq': hp.quniform('bagging_freq', 1, 7, 1),  # 0 제외 (의미 없음)
    
    # 리프 제약
    'min_data_in_leaf': hp.quniform('min_data_in_leaf', 10, 100, 5),  # 200은 너무 큼
    'min_child_samples': hp.quniform('min_child_samples', 5, 50, 5),  # 추가
    
    # 정규화
    'lambda_l1': hp.loguniform('lambda_l1', np.log(1e-8), np.log(10.0)),  # 로그 스케일
    'lambda_l2': hp.loguniform('lambda_l2', np.log(1e-8), np.log(10.0)),
    
    # 불균형 데이터 대응 (개선)
    'scale_pos_weight': hp.choice('scale_pos_weight', [
        1, 
        int(len(y_train) / sum(y_train)),  # 불균형 비율
        int(len(y_train) / sum(y_train)) * 0.5,  # 50%
        int(len(y_train) / sum(y_train)) * 1.5   # 150%
    ]),
    
    # 추가 파라미터 (성능 향상)
    'min_gain_to_split': hp.loguniform('min_gain_to_split', np.log(1e-5), np.log(1.0)),
    'reg_alpha': hp.loguniform('reg_alpha', np.log(1e-8), np.log(10.0)),  # L1
    'reg_lambda': hp.loguniform('reg_lambda', np.log(1e-8), np.log(10.0)),  # L2
}

# LGBM
lgbm_search_result = hyperopt_search(
    model_class  = LGBMClassifier,
    search_space = lgbm_search_space, 
    X_train     = X_train,
    y_train     = y_train,
    max_evals   = 100,  
    save_trials = True,
    verbose     = True
)

In [ ]:
best_params = lgbm_search_result['best_params']
best_params

In [ ]:
# 2단계: 최종 학습 및 평가

final_result = train_and_evaluate(
    model_class = LGBMClassifier,
    params      = best_params, 
    X_train     = X_train,
    y_train     = y_train,
    X_test      = X_test,
    y_test      = y_test,
    save_model  = True,
    verbose     = True
)

print("\n" + "=" * 70)
print("완료!")
print("=" * 70)
print(f"최종 결과: {final_result['result_dict']}")

results['sgd_ho_best'] = final_result['result_dict'] # 시각화용

In [ ]:
# oversampling한 것으로
final_result_over = train_and_evaluate(
    model_class = LGBMClassifier,
    params      = best_params, 
    X_train     = X_train,
    y_train     = y_train,
    X_test      = X_test,
    y_test      = y_test,
    save_model  = True,
    verbose     = True
)

print("\n" + "=" * 70)
print("완료!")
print("=" * 70)
print(f"최종 결과: {final_result_over['result_dict']}")

results['sgd_ho_best_over'] = final_result_over['result_dict'] # 시각화용

In [ ]:
# BestOpt 찾고 나서 스케일적용 버전 만들어서 모델링하기 
# StandardScaler 적용

X_train_sscaled, X_test_sscaled, scaler = pp.scale_data(X_train, X_test)

ss_result = train_and_evaluate(
    model_class = LGBMClassifier,
    params      = best_params, 
    X_train     = X_train_sscaled,
    y_train     = y_train,
    X_test      = X_test_sscaled,
    y_test      = y_test,
    save_model  = True,
    verbose     = True
)
results['sgd_ho_best_sscaled'] = ss_result['result_dict'] 

In [ ]:
# Data Scale2 
from sklearn.preprocessing import RobustScaler

rscaler = RobustScaler()
X_train_rscaled = rscaler.fit_transform(X_train)   # 학습 데이터로 fit + transform
X_test_rscaled = rscaler.transform(X_test)         # 테스트 데이터는 transform만

rs_result = uu.train_and_evaluate(
    model_class = LGBMClassifier,
    params      = best_params, 
    X_train     = X_train_rscaled,
    y_train     = y_train,
    X_test      = X_test_rscaled,
    y_test      = y_test,
    save_model  = True,
    verbose     = True
)
results['sgd_ho_best_rscaled'] = rs_result['result_dict']